# Phase 7 PCR — QKV control layer: Scaling + rotation


In [1]:
%pip install -q huggingface_hub tokenizers scikit-learn tqdm rouge-score


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, math, random, gc, tempfile, zipfile, uuid, re, hashlib
from pathlib import Path
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from huggingface_hub import hf_hub_download, HfFileSystem
import time
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
DATA_REPO = 'Exploration-Lab/CS-781-Capstone'
BUCKET_ROOT = 'hf://buckets/A-240031/legal-lm-phase1-bucket'
PHASE1_TAG = 'decoder_126M'
MODEL_PATH = 'checkpoints/decoder_126M/model.pt'
TOKENIZER_PATH = 'tokenizer_126M_decoder/tokenizer.json'
MODEL_CONFIG = {'d_model': 1024, 'n_heads': 8, 'n_kv_heads': 4, 'n_layers': 8, 'd_ff': 2816, 'max_seq_len': 2048}
MAX_SEQ_LEN = 512
USE_GRADIENT_CHECKPOINTING = False
SOURCE_SPANS = 8  # evenly spaced non-overlapping source passages
VAL_MAX_DOCS = 100  # Dev query subset for ranking-based selection; None = full Dev
LOG_EVERY_DOCS = 1000
USE_AMP = DEVICE == 'cuda'
AMP_DTYPE = torch.bfloat16 if USE_AMP and torch.cuda.is_bf16_supported() else torch.float16

def autocast_context():
    return torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP)

DATA_PREFIX = 'Phase_7_PCR'
assert 0 < MAX_SEQ_LEN <= MODEL_CONFIG['max_seq_len']
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
LR = 1e-3
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
NUM_EPOCHS = 5  # retained from the reference; lower this to cap training time
PATIENCE = 10
MIN_DELTA = 0.001
UPLOAD_CHECKPOINTS = True
TECHNIQUE = 'qkv_rotation_scaling'
TRAIN_FRACTION = 1.0  # use all training documents after holding out validation

# All loading/saving locations are controlled here.
# Paths below are relative to BUCKET_ROOT; an absolute hf:// path also works.
SAVE_TAG = None  # None generates a descriptive tag; or enter your own run tag.
SUBMISSION_ZIP_NAME = None  # None uses SAVE_TAG + '.zip'; or enter a filename.
CHECKPOINT_PREFIX = 'checkpoints'
RESUME = False  # True resumes SAVE_TAG; set SAVE_TAG to the existing latest tag.
WORK_ROOT = Path('/kaggle/working/pcr_runs') if Path('/kaggle/working').exists() else Path.cwd() / 'pcr_runs'

if RESUME and SAVE_TAG is None:
    raise ValueError('To resume, set SAVE_TAG to the existing run tag.')
if SAVE_TAG is None:
    SAVE_TAG = f'pcr_{TECHNIQUE}_{PHASE1_TAG}_len{MAX_SEQ_LEN}'
if not isinstance(SAVE_TAG, str) or not re.fullmatch(r'[A-Za-z0-9_.-]+', SAVE_TAG) or SAVE_TAG in {'.', '..'}:
    raise ValueError('SAVE_TAG must be a single name containing letters, numbers, _, - or .')
BEST_TAG = SAVE_TAG + '_best'
if SUBMISSION_ZIP_NAME is None:
    SUBMISSION_ZIP_NAME = SAVE_TAG + '.zip'
if Path(SUBMISSION_ZIP_NAME).name != SUBMISSION_ZIP_NAME or not SUBMISSION_ZIP_NAME.endswith('.zip'):
    raise ValueError('SUBMISSION_ZIP_NAME must be a filename ending in .zip.')
WORK_DIR = WORK_ROOT / SAVE_TAG
WORK_DIR.mkdir(parents=True, exist_ok=True)

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF')
    except Exception:
        pass
fs = HfFileSystem(token=HF_TOKEN)

def seed_everything(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_everything(SEED)
print('Device:', DEVICE)
def bucket_path(path):
    if path.startswith('hf://'):
        return path
    return BUCKET_ROOT.rstrip('/') + '/' + path.lstrip('/')

def checkpoint_remote_path(tag):
    return bucket_path(f"{CHECKPOINT_PREFIX.strip('/')}/{tag}/checkpoint.pt")

def download_bucket_file(remote_path):
    # Identity-specific cache directory; refresh the bytes on each call.
    remote = bucket_path(remote_path)
    cache_key = hashlib.sha256(remote.encode()).hexdigest()[:20]
    local = WORK_ROOT / '_inputs' / cache_key / remote.rsplit('/', 1)[-1]
    local.parent.mkdir(parents=True, exist_ok=True)
    temporary = local.with_name(local.name + '.download')
    fs.get_file(remote, str(temporary))
    os.replace(temporary, local)
    return local

def assert_run_destination():
    if RESUME:
        return
    for tag in [SAVE_TAG, BEST_TAG]:
        local = WORK_DIR / 'checkpoints' / tag / 'checkpoint.pt'
        if local.exists() or (UPLOAD_CHECKPOINTS and fs.exists(checkpoint_remote_path(tag))):
            raise FileExistsError(
                f'Checkpoint already exists for {tag}. Choose a new SAVE_TAG, '
                'or set RESUME=True with the original latest SAVE_TAG.'
            )
    manifest = experiment_metadata('PCR')
    manifest.update(save_tag=SAVE_TAG, best_tag=BEST_TAG, submission_zip=SUBMISSION_ZIP_NAME)
    with open(WORK_DIR / 'run_config.json', 'w') as f:
        json.dump(manifest, f, indent=2)

print('Phase-1 model:', bucket_path(MODEL_PATH))
print('Tokenizer:', bucket_path(TOKENIZER_PATH))
print('Latest checkpoint:', checkpoint_remote_path(SAVE_TAG))
print('Best checkpoint:', checkpoint_remote_path(BEST_TAG))
print('Submission ZIP:', WORK_DIR / SUBMISSION_ZIP_NAME)

NEGATIVES_PER_QUERY = 10
TEMPERATURE = 0.07
ENCODE_BATCH_SIZE = 1
IS_ENCODER_DECODER = False
IS_MOE = False

WARMUP_RATIO=0.05
WARMUP_START_FACTOR=0.1
LR_FACTOR=0.5
LR_BAD_CHECKS=2
LR_COOLDOWN_CHECKS=1
LR_MIN=1e-5
LR_LOSS_THRESHOLD=0.001
SCALE_ENABLED = True
ROTATION_ENABLED = True
CONTROL_LAST_N_BLOCKS = 8
LOG_SCALE_BOUND = math.log(2.0)
ANGLE_BOUND = math.pi / 4
assert SCALE_ENABLED or ROTATION_ENABLED
assert 0 < CONTROL_LAST_N_BLOCKS <= MODEL_CONFIG['n_layers']
assert LOG_SCALE_BOUND > 0 and 0 < ANGLE_BOUND <= math.pi
ADAPTER_CONFIG = dict(technique=TECHNIQUE,last_n_blocks=CONTROL_LAST_N_BLOCKS,
                      use_scale=SCALE_ENABLED,use_rotation=ROTATION_ENABLED,
                      log_scale_bound=LOG_SCALE_BOUND,angle_bound=ANGLE_BOUND,
                      controller='linear_head_dim_to_two',sharing='per_qkv_shared_across_heads_and_tokens',
                      placement='qk_after_rope_v_after_projection',rotation='same_angle_for_all_adjacent_pairs')

Device: cuda
Phase-1 model: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/decoder_126M/model.pt
Tokenizer: hf://buckets/A-240031/legal-lm-phase1-bucket/tokenizer_126M_decoder/tokenizer.json
Latest checkpoint: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/pcr_qkv_rotation_scaling_decoder_126M_len512/checkpoint.pt
Best checkpoint: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/pcr_qkv_rotation_scaling_decoder_126M_len512_best/checkpoint.pt
Submission ZIP: /kaggle/working/pcr_runs/pcr_qkv_rotation_scaling_decoder_126M_len512/pcr_qkv_rotation_scaling_decoder_126M_len512.zip


In [3]:
local_tokenizer = download_bucket_file(TOKENIZER_PATH)
tokenizer = Tokenizer.from_file(str(local_tokenizer))
tokenizer.no_padding()
tokenizer.no_truncation()
VOCAB_SIZE = tokenizer.get_vocab_size()
EMPTY_ID = tokenizer.token_to_id('<|eos|>')
PAD_ID = tokenizer.token_to_id('<|pad|>')
if EMPTY_ID is None:
    raise ValueError('Expected the Phase-1 <|eos|> token.')
if PAD_ID is None: PAD_ID = EMPTY_ID


EOS_ID = EMPTY_ID
BOS_ID = tokenizer.token_to_id('<|bos|>')
DECODER_START_ID = EOS_ID if BOS_ID is None else BOS_ID
PASSAGE_GAP_IDS = tokenizer.encode('\n[...]\n',add_special_tokens=False).ids
SOURCE_BUDGET = MAX_SEQ_LEN
print('Vocabulary:',VOCAB_SIZE,'| case token budget:',SOURCE_BUDGET)

(…)et/tokenizer_126M_decoder/tokenizer.json:   0%|          | 0.00/2.47M [00:00<?, ?B/s]

Vocabulary: 33467 | case token budget: 512


In [4]:
def download_json(filename):
    path = hf_hub_download(repo_id=DATA_REPO, filename=filename, repo_type='dataset', token=HF_TOKEN)
    with open(path, encoding='utf-8') as f: return json.load(f)

def case_text(value):
    if isinstance(value,str): return value
    if isinstance(value,list): return '\n'.join(case_text(x) for x in value)
    if isinstance(value,dict):
        for name in ('text','sentence','document'):
            if name in value: return case_text(value[name])
    raise ValueError('Expected case text as a string or a list of sentences')

def normalize_cases(raw,labelled=False):
    if isinstance(raw,list): entries=[(None,r) for r in raw]
    elif isinstance(raw,dict) and 'id' in raw: entries=[(None,raw)]
    elif isinstance(raw,dict): entries=list(raw.items())
    else: raise ValueError('Expected a list or ID-to-record mapping')
    records=[]; seen=set()
    for key,value in entries:
        if isinstance(value,(str,list)) and key is not None: value={'text':value}
        if not isinstance(value,dict): raise ValueError('Each case record must be an object')
        ident=value.get('id',value.get('case_id',value.get('query_id',key)))
        if not isinstance(ident,str): raise ValueError('Expected string case IDs; preserve leading zeros')
        if key is not None and ident!=key: raise ValueError('Case ID does not match mapping key')
        if ident in seen: raise ValueError(f'Duplicate case ID: {ident}')
        seen.add(ident)
        text=case_text(value.get('text',value.get('document'))).strip()
        if not text: raise ValueError(f'{ident}: empty case text')
        record={'id':ident,'text':text}
        if labelled:
            if 'relevant_candidates' not in value:
                raise ValueError(f'{ident}: missing relevant_candidates field; available fields: {list(value)}')
            gold=value['relevant_candidates']
            if not isinstance(gold,list) or any(not isinstance(c,str) for c in gold):
                raise ValueError(f'{ident}: relevant_candidates must be a list of string IDs')
            record['relevant_candidates']=list(dict.fromkeys(gold))
        records.append(record)
    if not records: raise ValueError('No cases found')
    return records

data={}; candidate_pools={}; fingerprints={}
for split,folder in [('train','Train'),('dev','Dev'),('test','Test')]:
    raw_queries=download_json(f'{DATA_PREFIX}/{folder}/{split}.json')
    raw_candidates=download_json(f'{DATA_PREFIX}/{folder}/{split}_candidates.json')
    data[split]=normalize_cases(raw_queries,labelled=split!='test')
    candidate_pools[split]=normalize_cases(raw_candidates)
    pool_ids={r['id'] for r in candidate_pools[split]}
    if len(pool_ids)<7: raise ValueError(f'{split}: fewer than seven candidates')
    for row in data[split]:
        missing=set(row.get('relevant_candidates',[]))-pool_ids
        if missing: raise ValueError(f'{split}/{row["id"]}: relevant IDs absent from this split pool: {sorted(missing)[:5]}')
    digest=hashlib.sha256()
    for records in (data[split],candidate_pools[split]):
        for row in records: digest.update(json.dumps(row,sort_keys=True,ensure_ascii=False).encode())
    fingerprints[split]=digest.hexdigest()
    print(split,'queries:',len(data[split]),'candidates:',len(pool_ids),
          'empty relevance lists:',sum(not r.get('relevant_candidates',[]) for r in data[split]) if split!='test' else 'private')
DATA_FINGERPRINT=fingerprints
train_rows=[r for r in data['train'] if r['relevant_candidates']]
print('Training queries with positives:',len(train_rows),'| skipped empty relevance lists:',len(data['train'])-len(train_rows))
if not train_rows: raise ValueError('No training queries have relevant candidates')
if TRAIN_FRACTION<1:
    train_rows=random.Random(SEED).sample(train_rows,max(1,round(len(train_rows)*TRAIN_FRACTION)))
dev_rows=data['dev'] if VAL_MAX_DOCS is None else data['dev'][:VAL_MAX_DOCS]

Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

train queries: 827 candidates: 4320 empty relevance lists: 10


dev.json:   0%|          | 0.00/5.87M [00:00<?, ?B/s]

... (3 outputs left out)

test.json:   0%|          | 0.00/9.34M [00:00<?, ?B/s]

Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

test queries: 237 candidates: 1727 empty relevance lists: private
Training queries with positives: 817 | skipped empty relevance lists: 10


In [5]:
def select_source_tokens(tokens,budget=SOURCE_BUDGET,spans=SOURCE_SPANS):
    if len(tokens)<=budget: return list(tokens)
    spans=min(spans,budget)
    content_budget=budget-(spans-1)*len(PASSAGE_GAP_IDS)
    if content_budget<spans: raise ValueError('Source budget too small for passage markers')
    widths=[content_budget//spans+int(i<content_budget%spans) for i in range(spans)]
    omitted=len(tokens)-content_budget
    if spans==1: return tokens[:budget]
    selected=[];consumed=0
    for i,width in enumerate(widths):
        start=consumed+i*omitted//(spans-1)
        if i:selected.extend(PASSAGE_GAP_IDS)
        selected.extend(tokens[start:start+width]);consumed+=width
    return selected

query_tokens={};candidate_tokens={}
for split in ('train','dev','test'):
    query_tokens[split]={r['id']:select_source_tokens(tokenizer.encode(r['text'],add_special_tokens=False).ids) or [EOS_ID]
                         for r in tqdm(data[split],desc=f'Tokenize {split} queries')}
    candidate_tokens[split]={r['id']:select_source_tokens(tokenizer.encode(r['text'],add_special_tokens=False).ids) or [EOS_ID]
                             for r in tqdm(candidate_pools[split],desc=f'Tokenize {split} candidates')}

class RetrievalDataset(Dataset):
    def __len__(self):return len(train_rows)
    def __getitem__(self,i):return train_rows[i]

def collate_retrieval(rows):return rows

def make_loader():
    return DataLoader(RetrievalDataset(),batch_size=BATCH_SIZE,shuffle=True,
                      collate_fn=collate_retrieval,num_workers=0)

train_candidate_ids=list(candidate_tokens['train'])
def sample_training_candidates(row):
    positive=random.choice(row['relevant_candidates']);gold=set(row['relevant_candidates'])
    if len(gold)>=len(train_candidate_ids):raise ValueError('Training query has no negative candidates')
    negatives=[]; used=set()
    for _ in range(NEGATIVES_PER_QUERY):
        # Rejection sampling avoids storing a candidate-pool copy for every query.
        for attempt in range(100):
            candidate=random.choice(train_candidate_ids)
            if candidate not in gold and candidate not in used:break
        else:
            eligible=[cid for cid in train_candidate_ids if cid not in gold and cid not in used]
            if not eligible:eligible=[cid for cid in train_candidate_ids if cid not in gold]
            candidate=random.choice(eligible)
        negatives.append(candidate);used.add(candidate)
    return [positive]+negatives

Tokenize train queries:   0%|          | 0/827 [00:00<?, ?it/s]

Tokenize train candidates:   0%|          | 0/4320 [00:00<?, ?it/s]

Tokenize dev queries:   0%|          | 0/118 [00:00<?, ?it/s]

Tokenize dev candidates:   0%|          | 0/1023 [00:00<?, ?it/s]

Tokenize test queries:   0%|          | 0/237 [00:00<?, ?it/s]

Tokenize test candidates:   0%|          | 0/1727 [00:00<?, ?it/s]

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# RoPE
# ============================================================

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len=2048, base=10000):
        super().__init__()

        inv_freq = 1.0 / (
            base ** (torch.arange(0, head_dim, 2).float() / head_dim)
        )

        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)

        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def forward(self, q, k):
        # q, k: [B, H, T, D]
        T = q.size(2)

        cos = self.cos[:T].unsqueeze(0).unsqueeze(0)
        sin = self.sin[:T].unsqueeze(0).unsqueeze(0)

        q1, q2 = q[..., ::2], q[..., 1::2]
        k1, k2 = k[..., ::2], k[..., 1::2]

        q_rot = torch.stack(
            [q1 * cos - q2 * sin,
             q1 * sin + q2 * cos],
            dim=-1,
        ).flatten(-2)

        k_rot = torch.stack(
            [k1 * cos - k2 * sin,
             k1 * sin + k2 * cos],
            dim=-1,
        ).flatten(-2)

        return q_rot, k_rot


# ============================================================
# SwiGLU
# ============================================================

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()

        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(
            F.silu(self.gate_proj(x)) * self.up_proj(x)
        )


# ============================================================
# GQA + SDPA
# ============================================================

class GQAAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len=2048):
        super().__init__()

        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0

        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(
            d_model,
            n_heads * self.head_dim,
            bias=False,
        )

        self.k_proj = nn.Linear(
            d_model,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.v_proj = nn.Linear(
            d_model,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.out_proj = nn.Linear(
            d_model,
            d_model,
            bias=False,
        )

        self.rope = RotaryEmbedding(
            self.head_dim,
            max_seq_len=max_seq_len,
        )

    def forward(self, x):

        B, T, _ = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B, T,
            self.n_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B, T,
            self.n_kv_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B, T,
            self.n_kv_heads,
            self.head_dim
        ).transpose(1, 2)

        q, k = self.rope(q, k)

        # Repeat KV heads for GQA
        repeat_factor = self.n_heads // self.n_kv_heads

        k = k.repeat_interleave(
            repeat_factor,
            dim=1,
        )

        v = v.repeat_interleave(
            repeat_factor,
            dim=1,
        )

        # PyTorch SDPA can dispatch to Flash Attention
        # / memory-efficient kernels automatically.
        attn_out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            is_causal=True,
        )

        attn_out = (
            attn_out
            .transpose(1, 2)
            .contiguous()
            .view(B, T, -1)
        )

        return self.out_proj(attn_out)


# ============================================================
# Decoder Block
# ============================================================

class DecoderBlock(nn.Module):
    def __init__(
        self,
        d_model,
        n_heads,
        n_kv_heads,
        d_ff,
        max_seq_len=2048,
    ):
        super().__init__()

        self.norm1 = nn.RMSNorm(d_model)
        self.attention = GQAAttention(
            d_model,
            n_heads,
            n_kv_heads,
            max_seq_len=max_seq_len,
        )

        self.norm2 = nn.RMSNorm(d_model)
        self.ffn = SwiGLU(
            d_model,
            d_ff,
        )

    def forward(self, x):

        x = x + self.attention(
            self.norm1(x)
        )

        x = x + self.ffn(
            self.norm2(x)
        )

        return x


# ============================================================
# Decoder-only Language Model
# ============================================================

class EfficientDecoderLM(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model=768,
        n_heads=12,
        n_kv_heads=4,
        n_layers=12,
        d_ff=3072,
        max_seq_len=2048,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model

        self.token_embedding = nn.Embedding(                             
            vocab_size,
            d_model,
        )

        self.blocks = nn.ModuleList([
            DecoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                n_kv_heads=n_kv_heads,
                d_ff=d_ff,
                max_seq_len=max_seq_len,
            )
            for _ in range(n_layers)
        ])

        self.final_norm = nn.RMSNorm(d_model)

        # Tied LM head
        self.lm_head = nn.Linear(
            d_model,
            vocab_size,
            bias=False,
        )

        self.lm_head.weight = self.token_embedding.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):

        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)

        elif isinstance(module, nn.Embedding):
            nn.init.xavier_uniform_(module.weight)

        elif isinstance(module, nn.RMSNorm):
            nn.init.ones_(module.weight)

    def forward(self, input_ids):

        x = self.token_embedding(input_ids)

        for block in self.blocks:
            x = block(x)

        x = self.final_norm(x)

        return self.lm_head(x)


In [7]:
class TwoOutputLinearController(nn.Module):
    """A linear map head_dim -> 2; no hidden layer or nonlinear feature network.

    Its two rows are stored separately so ablations can freeze the unused row.
    The output is [raw_log_scale, raw_rotation_angle] for each head/token vector.
    """
    def __init__(self,head_dim,use_scale=True,use_rotation=True):
        super().__init__()
        self.scale_weight=nn.Parameter(torch.zeros(head_dim),requires_grad=use_scale)
        self.scale_bias=nn.Parameter(torch.zeros(()),requires_grad=use_scale)
        self.angle_weight=nn.Parameter(torch.zeros(head_dim),requires_grad=use_rotation)
        self.angle_bias=nn.Parameter(torch.zeros(()),requires_grad=use_rotation)
    def forward(self,z):
        weight=torch.stack([self.scale_weight,self.angle_weight],dim=0)
        bias=torch.stack([self.scale_bias,self.angle_bias],dim=0)
        return F.linear(z,weight,bias)

class VectorScaleRotation(nn.Module):
    """One positive scalar and one repeated 2D rotation angle per head vector."""
    def __init__(self,head_dim,use_scale,use_rotation,log_scale_bound,angle_bound):
        super().__init__()
        if head_dim%2:raise ValueError('Pairwise rotation requires even head dimension')
        self.controller=TwoOutputLinearController(head_dim,use_scale,use_rotation)
        self.use_scale=use_scale;self.use_rotation=use_rotation
        self.log_scale_bound=log_scale_bound;self.angle_bound=angle_bound
    def forward(self,z):
        raw=self.controller(z).float()
        # Zero controller initialization gives exactly scale=1, angle=0.
        transformed=z.float()
        if self.use_rotation:
            theta=self.angle_bound*torch.tanh(raw[...,1])
            pairs=transformed.reshape(*transformed.shape[:-1],-1,2)
            x,y=pairs[...,0],pairs[...,1]
            cosine=torch.cos(theta)[...,None];sine=torch.sin(theta)[...,None]
            transformed=torch.stack([cosine*x-sine*y,sine*x+cosine*y],dim=-1).flatten(-2)
        if self.use_scale:
            log_scale=self.log_scale_bound*torch.tanh(raw[...,0])
            transformed=transformed*torch.exp(log_scale)[...,None]
        return transformed.to(z.dtype)

class ControlledQKVAttention(nn.Module):
    """Keep frozen GQA/RoPE; transform Q/K after RoPE and V after projection."""
    def __init__(self,attention,use_scale,use_rotation,log_scale_bound,angle_bound):
        super().__init__()
        self.original_attention=attention
        args=(attention.head_dim,use_scale,use_rotation,log_scale_bound,angle_bound)
        # Separate controllers avoid forcing the same rotation on every Q and K.
        # Each controller is shared across heads and tokens but depends on its input.
        self.q_transform=VectorScaleRotation(*args)
        self.k_transform=VectorScaleRotation(*args)
        self.v_transform=VectorScaleRotation(*args)
    def forward(self,x):
        a=self.original_attention;B,T,_=x.shape
        q=a.q_proj(x).view(B,T,a.n_heads,a.head_dim).transpose(1,2)
        k=a.k_proj(x).view(B,T,a.n_kv_heads,a.head_dim).transpose(1,2)
        v=a.v_proj(x).view(B,T,a.n_kv_heads,a.head_dim).transpose(1,2)
        q,k=a.rope(q,k)
        q=self.q_transform(q);k=self.k_transform(k);v=self.v_transform(v)
        # Adapt K/V before GQA repetition, preserving shared KV-head structure.
        repeat=a.n_heads//a.n_kv_heads
        k=k.repeat_interleave(repeat,dim=1);v=v.repeat_interleave(repeat,dim=1)
        output=F.scaled_dot_product_attention(q,k,v,is_causal=True)
        output=output.transpose(1,2).contiguous().view(B,T,-1)
        return a.out_proj(output)

from torch.utils.checkpoint import checkpoint as activation_checkpoint
from contextlib import contextmanager

class RetrievalBackbone(EfficientDecoderLM):
    def forward(self, input_ids):
        x = self.token_embedding(input_ids)
        for block in self.blocks:
            if self.training and USE_GRADIENT_CHECKPOINTING:
                x = activation_checkpoint(block, x, use_reentrant=False, preserve_rng_state=True)
            else: x = block(x)
        return self.lm_head(self.final_norm(x))




phase1_path=download_bucket_file(MODEL_PATH)

def load_phase1_backbone():
    model=RetrievalBackbone(VOCAB_SIZE,**MODEL_CONFIG)
    checkpoint=torch.load(phase1_path,map_location='cpu',weights_only=False)
    if 'model_state_dict' in checkpoint:state=checkpoint['model_state_dict']
    elif 'model' in checkpoint:state=checkpoint['model']
    elif 'state_dict' in checkpoint:state=checkpoint['state_dict']
    elif all(isinstance(v,torch.Tensor) for v in checkpoint.values()):state=checkpoint
    else:raise ValueError('Cannot identify Phase 1 weights')
    state=dict(state)
    if 'lm_head.weight' not in state:state['lm_head.weight']=state['token_embedding.weight']
    if not torch.equal(state['lm_head.weight'],state['token_embedding.weight']):
        raise ValueError('Phase 1 tied embedding/head weights disagree')
    model.load_state_dict(state,strict=True)
    del checkpoint,state
    model.requires_grad_(False)
    return model



def build_retrieval_model():
    model=load_phase1_backbone()
    for block in model.blocks[-CONTROL_LAST_N_BLOCKS:]:
        block.attention=ControlledQKVAttention(block.attention,SCALE_ENABLED,ROTATION_ENABLED,
                                             LOG_SCALE_BOUND,ANGLE_BOUND)
    model=model.to(DEVICE)
    assert model.lm_head.weight is model.token_embedding.weight
    print('QKV controller variant:',TECHNIQUE)
    print('Total/trainable parameters:',sum(p.numel() for p in model.parameters()),sum(p.numel() for p in model.parameters() if p.requires_grad))
    return model

def retrieval_hidden(model,ids,mask):
    if IS_ENCODER_DECODER:
        memory=model.encode(ids,encoder_mask=mask)
        mean=(memory.float()*mask[...,None]).sum(1)/mask.sum(1,keepdim=True).clamp_min(1)
        # A one-token decoder readout uses the pretrained decoder and cross-attention too.
        x=model.token_embedding(torch.full((len(ids),1),DECODER_START_ID,device=ids.device,dtype=torch.long))
        for block in model.decoder_blocks:
            if model.training and USE_GRADIENT_CHECKPOINTING:
                x,_=activation_checkpoint(block,x,memory,encoder_mask=mask,use_reentrant=False,preserve_rng_state=True)
            else:x,_=block(x,memory,encoder_mask=mask)
        return (mean+model.decoder_norm(x[:,0]).float())/2
    x=model.token_embedding(ids)
    for block in model.blocks:
        if IS_MOE:
            if model.training and USE_GRADIENT_CHECKPOINTING:
                x,_=activation_checkpoint(block,x,valid_mask=mask,use_reentrant=False,preserve_rng_state=True)
            else:x,_=block(x,valid_mask=mask)
        else:
            if model.training and USE_GRADIENT_CHECKPOINTING:
                x=activation_checkpoint(block,x,use_reentrant=False,preserve_rng_state=True)
            else:x=block(x)
    x=model.final_norm(x)
    return (x.float()*mask[...,None]).sum(1)/mask.sum(1,keepdim=True).clamp_min(1)

def embed_sequences(model,sequences):
    vectors=[]
    for start in range(0,len(sequences),ENCODE_BATCH_SIZE):
        part=sequences[start:start+ENCODE_BATCH_SIZE];width=max(map(len,part))
        ids=torch.full((len(part),width),PAD_ID,device=DEVICE,dtype=torch.long)
        mask=torch.zeros_like(ids,dtype=torch.bool)
        for i,seq in enumerate(part):
            ids[i,:len(seq)]=torch.tensor(seq,device=DEVICE);mask[i,:len(seq)]=True
        with autocast_context():v=retrieval_hidden(model,ids,mask)
        vectors.append(F.normalize(v.float(),dim=-1))
    return torch.cat(vectors)

def retrieval_loss_sum(model,rows):
    total=None
    for row in rows:
        candidates=sample_training_candidates(row)
        sequences=[query_tokens['train'][row['id']]]+[candidate_tokens['train'][cid] for cid in candidates]
        embeddings=embed_sequences(model,sequences)
        scores=(embeddings[:1]@embeddings[1:].T)/TEMPERATURE
        loss=F.cross_entropy(scores,torch.zeros(1,dtype=torch.long,device=DEVICE))
        total=loss if total is None else total+loss
    return total,len(rows)

@torch.no_grad()
def rank_cases(model,split,rows=None):
    was_training=model.training;model.eval()
    try:
        if rows is None:rows=data[split]
        ids=list(candidate_tokens[split]);embeddings=[];started=time.perf_counter()
        for start in tqdm(range(0,len(ids),ENCODE_BATCH_SIZE),desc=f'Encode {split} pool'):
            batch=ids[start:start+ENCODE_BATCH_SIZE]
            embeddings.append(embed_sequences(model,[candidate_tokens[split][cid] for cid in batch]).cpu())
        matrix=torch.cat(embeddings);predictions={};hits=gold_count=0
        for row in tqdm(rows,desc=f'Rank {split}'):
            q=embed_sequences(model,[query_tokens[split][row['id']]])[0].cpu()
            # Sorting stable ties follows the candidate file order.
            selected=torch.argsort(matrix@q,descending=True,stable=True)[:7].tolist()
            predictions[row['id']]=[ids[i] for i in selected]
            if split!='test':
                gold=set(row['relevant_candidates']);gold_count+=len(gold)
                hits+=len(gold.intersection(predictions[row['id']]))
        p=hits/(7*len(rows)) if rows else 0.0;r=hits/gold_count if gold_count else 0.0
        metrics=dict(micro_precision_at_7=p,micro_recall_at_7=r,micro_f1_at_7=2*p*r/(p+r) if p+r else 0.0,
                     hits=hits,gold_relevant=gold_count,queries=len(rows),ranking_seconds=time.perf_counter()-started)
        return predictions,metrics
    finally:model.train(was_training)

(…)bucket/checkpoints/decoder_126M/model.pt:   0%|          | 0.00/515M [00:00<?, ?B/s]

In [8]:
def experiment_metadata(domain='PCR'):
    # Epoch target is deliberately not part of compatibility: it may be extended on resume.
    return dict(domain=domain,phase1_tag=PHASE1_TAG,model_path=MODEL_PATH,tokenizer_path=TOKENIZER_PATH,
                model_config=MODEL_CONFIG,adapter_config=ADAPTER_CONFIG,technique=TECHNIQUE,
                source_budget=SOURCE_BUDGET,source_spans=SOURCE_SPANS,data_fingerprint=DATA_FINGERPRINT,
                batch_size=BATCH_SIZE,grad_accum_steps=GRAD_ACCUM_STEPS,lr=LR,weight_decay=WEIGHT_DECAY,
                negatives_per_query=NEGATIVES_PER_QUERY,temperature=TEMPERATURE,seed=SEED,
                train_fraction=TRAIN_FRACTION,val_max_docs=VAL_MAX_DOCS,
                scheduler=dict(warmup_ratio=WARMUP_RATIO,start_factor=WARMUP_START_FACTOR,factor=LR_FACTOR,
                               bad_checks=LR_BAD_CHECKS,cooldown=LR_COOLDOWN_CHECKS,min_lr=LR_MIN,threshold=LR_LOSS_THRESHOLD))

def save_retrieval_checkpoint(tag, model, optimizer, epoch, extra, domain):
    folder = WORK_DIR / 'checkpoints' / tag
    folder.mkdir(parents=True, exist_ok=True)
    rng = {'python': random.getstate(), 'torch': torch.get_rng_state()}
    if torch.cuda.is_available(): rng['cuda'] = torch.cuda.get_rng_state_all()
    payload = dict(model=model.state_dict(), optimizer=optimizer.state_dict(), rng=rng,
                   epoch=epoch, extra=extra, metadata=experiment_metadata(domain))
    tmp = folder / 'checkpoint.tmp'
    torch.save(payload, tmp)
    os.replace(tmp, folder / 'checkpoint.pt')
    if UPLOAD_CHECKPOINTS:
        fs.put_file(str(folder / 'checkpoint.pt'), checkpoint_remote_path(tag))

def load_retrieval_checkpoint(tag, model, domain, optimizer=None, restore_rng=False):
    path = WORK_DIR / 'checkpoints' / tag / 'checkpoint.pt'
    if not path.exists():
        if UPLOAD_CHECKPOINTS:
            path = download_bucket_file(checkpoint_remote_path(tag))
        elif not path.exists():
            raise FileNotFoundError(path)
    payload = torch.load(path, map_location='cpu', weights_only=False)  # own trusted checkpoints only
    if payload['metadata'] != experiment_metadata(domain):
        changed = [k for k in set(payload['metadata']) | set(experiment_metadata(domain)) if payload['metadata'].get(k) != experiment_metadata(domain).get(k)]
        raise ValueError('Checkpoint configuration mismatch: ' + ', '.join(sorted(changed)))
    model.load_state_dict(payload['model'])
    if optimizer is not None: optimizer.load_state_dict(payload['optimizer'])
    if restore_rng:
        random.setstate(payload['rng']['python'])
        torch.set_rng_state(payload['rng']['torch'])
        if torch.cuda.is_available() and 'cuda' in payload['rng']:
            torch.cuda.set_rng_state_all(payload['rng']['cuda'])
    return payload



class ValidationLRScheduler:
    def __init__(self, optimizer, total_updates):
        assert total_updates > 0 and 0 < WARMUP_RATIO < 1
        assert 0 < WARMUP_START_FACTOR <= 1 and 0 < LR_FACTOR < 1
        assert LR_BAD_CHECKS >= 1 and 0 <= LR_MIN <= LR
        self.optimizer = optimizer
        self.updates = 0
        self.warmup_updates = max(1, math.ceil(total_updates * WARMUP_RATIO))
        self.peak_lrs = [g['lr'] for g in optimizer.param_groups]
        # PyTorch reduces after num_bad_epochs > patience: 1 means two bad checks.
        self.plateau = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=LR_FACTOR, patience=LR_BAD_CHECKS-1,
            threshold=LR_LOSS_THRESHOLD, threshold_mode='rel',
            cooldown=LR_COOLDOWN_CHECKS, min_lr=LR_MIN)
        self._set_warmup_lr()

    def _set_warmup_lr(self):
        factor = WARMUP_START_FACTOR + (1-WARMUP_START_FACTOR) * self.updates/self.warmup_updates
        for group, peak in zip(self.optimizer.param_groups, self.peak_lrs):
            group['lr'] = peak * factor

    def optimizer_step_completed(self):
        self.updates += 1
        if self.updates <= self.warmup_updates: self._set_warmup_lr()

    def validation_step(self, loss):
        if not math.isfinite(loss): raise ValueError('Non-finite validation loss')
        # Warmup and plateau never fight over the learning rate.
        if self.updates >= self.warmup_updates: self.plateau.step(loss)

    def state_dict(self):
        return dict(updates=self.updates, warmup_updates=self.warmup_updates,
                    peak_lrs=self.peak_lrs, plateau=self.plateau.state_dict(),
                    current_lrs=[g['lr'] for g in self.optimizer.param_groups])

    def load_state_dict(self, state):
        self.warmup_updates = state['warmup_updates']  # restore the original horizon when extending epochs
        self.updates = state['updates']
        self.peak_lrs = state['peak_lrs']
        self.plateau.load_state_dict(state['plateau'])
        for group, lr in zip(self.optimizer.param_groups, state['current_lrs']): group['lr'] = lr

In [9]:
def train_retrieval(domain='PCR'):
    assert_run_destination();seed_everything(SEED)
    model=build_retrieval_model()
    params=[p for p in model.parameters() if p.requires_grad]
    optimizer=torch.optim.AdamW(params,lr=LR,weight_decay=WEIGHT_DECAY)
    scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP and AMP_DTYPE==torch.float16)
    loader=make_loader();updates_per_epoch=math.ceil(len(loader)/GRAD_ACCUM_STEPS)
    scheduler=ValidationLRScheduler(optimizer,NUM_EPOCHS*updates_per_epoch)
    start=0;extra=dict(best_f1=-1.0,patience_reference=-1.0,stale=0,history=[],best_tag=BEST_TAG)
    if RESUME:
        payload=load_retrieval_checkpoint(SAVE_TAG,model,domain,optimizer,restore_rng=True)
        start=payload['epoch'];extra=payload['extra']
        scaler.load_state_dict(extra['scaler']);scheduler.load_state_dict(extra['scheduler'])
        del payload
        print('Resume at completed epoch:',start,'| target epochs:',NUM_EPOCHS)
    for epoch in range(start,NUM_EPOCHS):
        if extra['stale']>=PATIENCE:break
        model.train();optimizer.zero_grad(set_to_none=True)
        train_sum=train_n=group_n=seen=0
        for step,rows in enumerate(tqdm(loader,desc=f'PCR epoch {epoch+1}')):
            loss_sum,count=retrieval_loss_sum(model,rows)
            if not torch.isfinite(loss_sum):raise FloatingPointError('Nonfinite retrieval loss')
            scaler.scale(loss_sum).backward()
            train_sum+=loss_sum.item();train_n+=count;group_n+=count;seen+=count
            if (step+1)%GRAD_ACCUM_STEPS==0 or step+1==len(loader):
                scaler.unscale_(optimizer)
                for p in params:
                    if p.grad is not None:p.grad.div_(group_n)
                torch.nn.utils.clip_grad_norm_(params,GRAD_CLIP)
                previous=scaler.get_scale();scaler.step(optimizer);scaler.update()
                if scaler.get_scale()>=previous:scheduler.optimizer_step_completed()
                optimizer.zero_grad(set_to_none=True);group_n=0
            if seen and seen%LOG_EVERY_DOCS<len(rows):print('Queries:',seen,'| loss:',train_sum/train_n)
        predictions,metrics=rank_cases(model,'dev',dev_rows)
        f1=metrics['micro_f1_at_7'];scheduler.validation_step(1.0-f1)
        improved=f1>extra['best_f1']
        if improved:extra['best_f1']=f1;extra['best_metrics']=metrics
        if f1>extra['patience_reference']+MIN_DELTA:extra['patience_reference']=f1;extra['stale']=0
        else:extra['stale']+=1
        extra['history'].append(dict(epoch=epoch+1,train_loss=train_sum/train_n,lr=optimizer.param_groups[0]['lr'],**metrics))
        extra['scaler']=scaler.state_dict();extra['scheduler']=scheduler.state_dict()
        print('Dev retrieval:',metrics)
        if improved:save_retrieval_checkpoint(BEST_TAG,model,optimizer,epoch+1,extra,domain)
        save_retrieval_checkpoint(SAVE_TAG,model,optimizer,epoch+1,extra,domain)
        with open(WORK_DIR/'PCR_history.json','w') as f:json.dump(extra['history'],f,indent=2)
    if not extra['history']:raise ValueError('No trained checkpoint available; NUM_EPOCHS must be positive')
    load_retrieval_checkpoint(BEST_TAG,model,domain)
    with open(WORK_DIR/'PCR_best_dev_metrics.json','w') as f:json.dump(extra['best_metrics'],f,indent=2)
    # Retain latest for exact continuation; it differs from the best validation epoch.
    del optimizer
    return model,extra['history']

In [10]:
def validate_predictions(predictions,documents,pool):
    if set(predictions)!={row['id'] for row in documents}:raise ValueError('Prediction query IDs do not match Test queries')
    allowed={row['id'] for row in pool}
    for qid,ids in predictions.items():
        if len(ids)!=7 or len(set(ids))!=7:raise ValueError(f'{qid}: expected seven unique IDs')
        if not set(ids)<=allowed:raise ValueError(f'{qid}: prediction outside the Test pool')

model,history=train_retrieval()
predictions,_=rank_cases(model,'test')
validate_predictions(predictions,data['test'],candidate_pools['test'])
with open(WORK_DIR/'prediction.json','w',encoding='utf-8') as f:json.dump(predictions,f,ensure_ascii=False,indent=2)

QKV controller variant: qkv_rotation_scaling
Total/trainable parameters: 128665648 6192


PCR epoch 1:   0%|          | 0/409 [00:00<?, ?it/s]

Encode dev pool:   0%|          | 0/1023 [00:00<?, ?it/s]

Rank dev:   0%|          | 0/100 [00:00<?, ?it/s]

Dev retrieval: {'micro_precision_at_7': 0.23857142857142857, 'micro_recall_at_7': 0.21548387096774194, 'micro_f1_at_7': 0.2264406779661017, 'hits': 167, 'gold_relevant': 775, 'queries': 100, 'ranking_seconds': 60.402657020000106}


Processing Files (0 / 0): |          |  0.00B /  0.00B            

... (31 outputs left out)

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

Encode test pool:   0%|          | 0/1727 [00:00<?, ?it/s]

Rank test:   0%|          | 0/237 [00:00<?, ?it/s]

In [11]:
# Packaging can be rerun without retraining.
json_path=WORK_DIR/'prediction.json'
with open(json_path,encoding='utf-8') as f:predictions=json.load(f)
validate_predictions(predictions,data['test'],candidate_pools['test'])
zip_path=WORK_DIR/SUBMISSION_ZIP_NAME
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:z.write(json_path,arcname='prediction.json')
print('Submission:',json_path,'| ZIP:',zip_path)

Submission: /kaggle/working/pcr_runs/pcr_qkv_rotation_scaling_decoder_126M_len512/prediction.json | ZIP: /kaggle/working/pcr_runs/pcr_qkv_rotation_scaling_decoder_126M_len512/pcr_qkv_rotation_scaling_decoder_126M_len512.zip
